## Part 0: Introduction and preparation

Welcome to the Large Language Model (LLM) project of ELEN90088!

In this project, you will transition from a consumer of AI to an engineer who understands the mechanics of LLMs. You will explore how these models are loaded, manipulated, trained and optimized to run on GPUs.

This project focuses on implementation and engineering. While the theories behind the transformer architecture (Attention, LayerNorm etc.) may be covered in lectures, you will not be asked to implement
these components from scratch (note that we may ask you to explain attention mechnaisms from a high level, but not in detail). Instead, you will learn to use the HuggingFace library to leverage these components.

This project is structured into six parts. You will follow the notebook to get familiar with LLMs in part 1-5, and design your own mini-project in part 6.

Specifically, this project includes the following parts:
1. The `pipeline`: Use high-level LLM APIs to perform tasks like sentiment analysis and text generation.
2. Behind the `pipeline`: Understand and manage the key components of LLM workflow, including tokenizers, models and datasets.
3. Inference: Use the tools from part 2 to implement the `pipeline` in part 1.
4. Efficient LLM - Model Compression: Reduce the LLM size while maintaining the LLM performance.
5. Efficient LLM - Parameter-efficient Fine-Tuning (PEFT): Use advanced methods to specialize a general-purpose model for specific tasks.
6. Mini-project: Design and implement your own LLM project with the skills you have learned from part 1-5.

Notes:
1. GPU usage: for part 1-2, cpu may be sufficient (loading models for a few minutes is normal). From part 3, you will need access to GPU (Google Colab, and/or Windows with Nvidia) or MPS (on Mac).
2. Project report: This notebook includes some questions that help you understand more details about LLM and implemetation. These questions are labelled as "**Questions (for project report)**". You may prepare the responses to these questions as part of your report.
3. Get help: While this notebook was prepared to explain as much terminology as possible, you may find some definitions unfamilar and need time to process. This is perfectly normal especially for beginners. You are encouraged to find other guidelines, technical reports and blogs from Google and/or AI. For example, what is a BERT model and what does it do? We will test your understanding during oral exams.
4. Code: Most of the code in part 1-5 has been provided to you. We also left some parts for you to implement as exercises.
   
    That said, instead of just running the cells, we highly recommend you to understand **why** the code works, and how you could modify some parameters, variables to improve the code performance. We will test whether you **really** understand the code during oral exams. 
5. (**Important**) Mistral vs. Phi LLM. As you will see, starting from part 2, this notebook frequently talks about the Mistral model, and the code is often about the Mistral model. While Mistral has great functionalities, it may not be feasible to run Mistral models on your hardware, which is perfectly fine becuase there are other models that can run on your hardware.

    If you are using standard hardware (eg. Macbook Pro, or Colab free tier), we recommend you to use Phi model from Microsoft, which we have tested and confirmed it will run on your hardware and the free version of Google Colab. The only change you need to make is to ensure you are loading and using Phi model (not Mistral which may crash your code) where necessary. 
    
    For the tasks that you may use Phi model and update the code (written for Mistral models), we have added a note: "**(You can use Phi model for this task, please modify the code if requried)**" 

    Your project marks are **not** affected by which model you use.

6. Oral exam: We will test your understanding of this notebook, including but not limited to (your answers to) "**Questions (for project report)**", code details, and your mini project during oral exams.

In [ ]:
import os
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from datasets import load_dataset
import pandas as pd

In [ ]:
# Define the device selector
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
import sys
print(f"Python is running from: {sys.executable}")
print(f"Torch is installed in: {torch.__file__}")

In [ ]:
# uncomment the line below to check GPU status (not examinable)
# !nvidia-smi

In [ ]:
# Some checks to make sure the project environment will work (not examinable)

from huggingface_hub import split_torch_state_dict_into_shards
print("Successfully imported!")

In [ ]:
# Some checks to make sure the project environment will work (not examinable)

import transformers
from packaging import version

required_version = "4.34.0"
current_version = transformers.__version__

if version.parse(current_version) < version.parse(required_version):
    print(f"❌ ERROR: Your transformers version ({current_version}) is too old!")
    print(f"Please run: pip install -U transformers")
else:
    print(f"✅ SUCCESS: Transformers version {current_version} is compatible with Mistral.")

## Part 1: Warm up, LLM pipeline

In this part, we will treat LLMs as a black box. We will use the Hugging Face `pipeline` API, which abstracts the complexities of tokenization, model architecture and output decoding into a single line of code.

### Example 1a: Autoregressive Text Generation

Unlike classification in other neurnal network applications, text generation is a generative task. The model predicts possible next token based on all previous tokens in the sequence.

In [ ]:
generator = pipeline("text-generation")
generator("In this LLM project, we will teach you how to")

**Questions (for project report)**

- Did you get some warning messages or errors? If the warning messages (or notes) did not stop you from running the cell above, what could you do to eliminate the warning?

**Hint**: you may want to specify a model, explain `gpt2` model, and understand `pad_token_id`, `eos_token_id`. You may also change the string in `generator` and discuss the quality of generated text, does the output text make sense?

### Example 1b: Sentiment analysis

The goal of sentiment analysis is to claassify a string of text into a category (eg. Positive or Negative). This is a discriminative task, not a text generation task.

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier("How good is ELEN90088")

**Questions (for project report)**

You will see something like "No model was supplied, defaulted to **distilbert**/distilbert-base-uncased-finetuned-sst-2-english"

- What's BERT, how's that different from other models (eg. GPT-2)?

- What's `distilbert`? What's `distilbert-base-uncased-finetuned-sst-2-english`?

- What are `label` and `score`? Can you try different sentiment and check the output `label` and `score`?

### Exploring `pipeline`

Hugging Face offers dozens of pre-configured pipelines for differentt modalities. Now that you have seen how `pipeline()` works (twice), it is your turn to explore.

### Exercise:

Choose another two `pipeline()` tasks (that's not sentiment analysis or text generation) and implement them in the cells below. Also briefly explain the tasks and their performance in your report. For example, does the model output meet your expectations?

**Hint**: This link might help. https://huggingface.co/docs/transformers/en/main_classes/pipelines

In [ ]:
# pipeline() Task 1
# Your code here

In [ ]:
# pipeline() Task 2
# Your code here

**Questions (for project report)**

- While the `pipeline()` API is easy to use (two lines of code), it hides several critical steps of LLM workflow. Based on your observation of execution time, what do you think is happening when you call the `pipeline()` function for the first time vs. the second time? Briefly explain in your report.

## Part 2: Behind the pipeline - LLM workflow

While the `pipeline()` API is convenient, it abstracts away the engineering decisions required for high-performance systems. To optimize for memory and/or speed, we must gain granular control over the individual components of the LLM workflow. Key components in the workflow include the tokenizer, the model and the datasets.

#### The choice of model
For this part, we are using Mistral-7B-Instruct-v0.3 for a few reasons:

- It is widely considered one of the best models in the "7-Billion parameter" class, outperforming some models twice its size.
- Since it has been fine-tuned to follow instructions, it is highly capable of performing zero-shot and few shot tasks we will attempt in Part 3.

**Note:** For part 2, you should be able to load Mistral model with any hardware. However when you start inference in next parts, your hardware may not be able to handle Mistral. Don't panic, you can use Phi model instead.
  
#### Manage storage (**Optional**)

When you download an LLM, you are downloading massive files. For example, the Mistral weights alone are around 14.5 GB. By default, Hugging Face saves these in a hidden folder in your Home directory (eg. `~/.cache/huggingface` on Mac OS). This could become tricky if you download a few models and datasets without checking and managing disk storage. A simple method to get easy access to the downloaded files (and manage them) is to create a cache folder within your project directory, and download all the project files in this dedicated cache folder.

**Note:** While this is recommended, setting up this cache folder is optional and will not affect your project marks.

In [ ]:
cache_dir = "./project_cache"

**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3" #we use Mistral instruct model for this part

# Uncomment below if you want to use Phi model: it's easier to run on local device and free Colab!
# model_id = "microsoft/Phi-3.5-mini-instruct"
# cache_dir = "./phi_project"

#### 2.1 The tokennizer

Computers process numbers, not text. The tokenizer acts as the "translator" between human language and the high-dimensional vectors the model understands. It performs three critical steps:

1. Normalization: Removing extra spaces, lowercase conversion, etc.
2. Pre-tokenization: Splitting text intowords or sub-words.
3. Encoding: Mapping those sub-words to their unique integer IDs in the model's vocabulary.

It is a fundamental requirement that your tokenizer and model come from the exact same "designer" and version (in this part, we use the Mistral model). If they don't match, for example, we load a Llama-3 tokenizer with a Mistral model, the system will not necessaricly throw a code error, but the results will likely be ungrammatical texts. Every model is trained with a specific Vocabulary index. If a tokenizer tells the Mistral model that the ID `5678` means "Apple", but the Mistral model and tokenizer was trained to believe `5678` means "float", then the mathematical logic of the network will collapse.

#### Exercise:

Load the tokenizer and observe how it handles the Mistral/Phi vocabulary.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=False,
    cache_dir=cache_dir # Note that we specified the cache folder here and in the next parts where we load models, tokenizers etc.
)


text = "Optimization is essential for LLMs."
ids = tokenizer.encode(text)

print(f"Token IDs: {ids}")
print(f"Decoded: {tokenizer.decode(ids)}")

# more examples...
print(f"Decoded: {tokenizer.decode(5678)}")

In [ ]:
# more examples...
text = "Tokenization"
ids = tokenizer.encode(text)

print(f"Token IDs: {ids}")
print(f"Decoded: {tokenizer.decode(ids)}")

print(f"Decoded: {tokenizer.decode(17393)}")

**Questions (for project report)**

- A token can be a word, or part of a word. For example, "tokenization" includes "token" and "ization". Can you find other words that consist of multiple tokens?

- Can you explain why tokenization may be a good approach to turn texts into numbers? **Hint:** What would happen if every word was represented by an unique integer?

#### 2.2 The model

In the Hugging Face ecosystem, "the model" is the computational heart of the system that has a collection of billions of pre-trained weights organized into a specific neural architecture. While the tokeizer handles the text-to-ID translation, the model performs the massive matrix multiplication required to predict the probability of the next token in a sequence.

LLMs typically store weights using floating-point numbers. While standard software often use 32-bit (Float32), moden LLMs utilize 16-bit formats like Float16. This format provides the same dynamic range as Float32 but with half the memory footprint, offering a crucial balance between numerical stability and hardware efficiency. We will dive deeper into the these data types and how they reduce model size in part 4.

#### Exercise:

Run the following code to load the model.


**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
print("Loading raw model (16-bit baseline)...")
model_raw = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=cache_dir
)

print(f"Model successfully loaded on: {model_raw.device}")
print(f"Model precision: {model_raw.dtype}")

**Questions (for project report)**

- How can you confirm whether your model is loaded on CPU or GPU?

- Explore model architecture (attention heads, sliding window etc.). It's perfectly fine to use online resources to understand the architecture (please cite your sources). It's even better to verify the online resources by running some simple code to show the structure (eg. layer names) of the loaded model.

#### Save and load from local path

In professional workflows, you often need to save a specific state of a model to ensure reproducibility or to move it between computing nodes without relying on an internet connection. By using `save_pretrained()`, you create a local snapshot of the model weights and configuration files. We will use this path again in part 4.

**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
# Save the model and configuration to a local directory
local_path = "./mistral_16bit_save"

model_raw.save_pretrained(local_path)
tokenizer.save_pretrained(local_path)

To load the model from your saved files instead of downloading (every time) from the Hugging Face hub, simply replace the `model_id` with your local directory path. The library will detect that you are pointing to a folder rather than a remote repository name.

In [ ]:
# To reload the model later from your local storage:
model_local = AutoModelForCausalLM.from_pretrained(
    local_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer_local = AutoTokenizer.from_pretrained(local_path)

#### Clean up memory (**Optional**)

The function below can clean up your GPU memory so that you can load and try new models without receiving 'out-of-memory' errors.

In [ ]:
import gc
import torch

def reset_gpu_memory(model_variable_name='model'):
    """
    Fully clears the GPU VRAM by deleting the model and flushing the
    PyTorch cache. Use this before switching between different model types.
    """
    # 1. Access the global variable for the model
    if model_variable_name in globals():
        print(f"--- Deleting {model_variable_name} from memory ---")
        del globals()[model_variable_name]
    else:
        print(f"--- No variable named '{model_variable_name}' found ---")

    # 2. Force Python's Garbage Collector to run
    gc.collect()

    # 3. Clear the PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize() # Wait for all kernels to finish
        print("VRAM successfully cleared.")
    else:
        print("No GPU detected to clear.")

# --- How to use it in the Lab ---
# After finishing Part 2 (16-bit):
reset_gpu_memory('model_raw')

# Now you can safely load the 4-bit model in Part 4
# model = AutoModelForCausalLM.from_pretrained(...)

#### Check GPU memory (**Optional**)

The function below can check your GPU resources. Note that we recommended to use your own device (if it has a GPU) and/or FEIT GPU Desktop. 

If you use the free version of Google Colab you will see the allocated and reserved memory are very low, which would not be suffifient for this project. 

In [ ]:
import torch

def print_gpu_utilization():
    if torch.cuda.is_available():
        # Returns memory in bytes, so we convert to GB
        used_memory = torch.cuda.memory_allocated() / 1024**3
        reserved_memory = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory Allocated: {used_memory:.2f} GB")
        print(f"GPU Memory Reserved: {reserved_memory:.2f} GB")
    else:
        print("No GPU detected.")

# Check before loading Mistral
print_gpu_utilization()

#### 2.3 A note on datasets

While we have loaded our tokenizer and model, a LLM workflow in incomplete without data. You may have noticed we have not included the code to process any datasets in this part. We will introduce the dataset in part 3, where we will write the logic to feed data into the model.

## Part 3: Inference Engineering

In this part, you will use the tokenizer and the model you loaded in part 2 to implement the `pipeline` tasks from part 1.

By handling the tensors, you will observe the "autoregressive" nature of LLMs: how they predict the next token based on the previous context, and how prompt engineering can turn a general-purpose text generator into a specialized classifier.

### Part 3a: Text response with LLM

In example 1a, the `pipeline` handled the iteration of predicting word after word. Now, you will implement this manually.

For part 3a, you will learn to use the `model.generate()` function, which gives you control over variables like `temperature` (creativity) and `max new tokens` (output length).

**(You can use Phi model for this task, please modify the code if requried)**

In particular, you would want to search the correct instruction formatr for Phi model, which is different from Mistral.

In [ ]:
# 1. Setup - Use the same ID and directory from previous tasks
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
cache_dir = "./mistral_project"

# Uncomment below if you want to use Phi model: it's easier to run on local device and free Colab!
# You may have small issues to run the code below with Phi model but they can be quickly fixed (no need to panic at all). You can search on Google, use AI, or talk to demonstrators.
# model_id = "microsoft/Phi-3.5-mini-instruct"
# cache_dir = "./phi_project"

# 2. Load the Components
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Baseline precision
    device_map="auto",
    cache_dir=cache_dir
)

def generate_text(user_prompt, max_tokens=100, temperature=0.7):
    """
    Manually handles the prompt encoding, generation, and decoding.
    """
    # Wrap in Mistral's required instruction format [INST] [/INST]
    # What's Phi's instruction format?
    formatted_prompt = f"[INST] {user_prompt} [/INST]"

    # Step A: Tokenization (Text -> IDs)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    # Step B: Generation
    # We use 'do_sample=True' to allow for creativity via Temperature
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Step C: Decoding (IDs -> Text)
    # We slice the output to remove the original prompt from the display
    new_tokens = output_tokens[0][len(inputs["input_ids"][0]):]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response

In [ ]:
# --- Test the Function ---
my_prompt = "Outline an introduction to LLM for students with some maths and programming background"
print(f"Prompt: {my_prompt}\n")
print(f"Response:\n{generate_text(my_prompt, max_tokens=1000)}") # With max_tokens=1000, you may need to wait more than 5 minutes to get the output. You can set a smaller max_tokens first, confirm the code is working, and then switch to larger max_tokens.

**Questions (for project report)**

- How does the output relate to the prompt? Does it talk about LLM as in 'Large Language Models'? If you run the above code again, does the output change?

- How could you make sure the output talks about Large Language Models?

- In step B of `generate_text` function, there are a few variables that you can adjust. 
    - Explain `max_new_tokens`, `temperature`, `do_sample`.
    - Change these variables and observe the output.

### Part 3b: Sentiment Analysis with LLM

In example 1b, you used a BERT-based model for sentiment analysis. BERT is an encoder-only model designed to output a numerical score for a fixed set of labels.

In part 3b, we take a different approach using a generative LLM. We don't use a classification "head" in LLM architecture, instead, we use prompts. By providing the model with a clear instruction (and optionally a few examples), we "steer" the model to output exactly one word: `Positive` or `Negative` (sentiment).

Unlike BERT, an LLM can technically say whatever it wants. If your prompt is vague, the LLM might explain why it thinks an input is positive instead of just giving the positive/negative label. You will explore how to constrain the output to ensure it fits back into your workflow.

**Note**: While BERT is efficient for simple classification, the industry is shifting toward generative inference, which can achieve:
- Zero-shot capabilities: How a model can perform a task it wasn't explicit trained for.
- Instruction following: How the `[INST]` and `[/INST]` tags change the probability distribution of the next token.
- Unified architecture: the power of using one single model for both generation and classification.

#### Exercise: Simple sentiment analysis

Fill in the code below. Use Mistral and text generation function to complete sentiment analysis tasks. **Hint**: Check the code in part 3a.

**(You can use Phi model for this task, please modify the code if requried)**

In particular, you would want to search the correct instruction formatr for Phi model, which is different from Mistral.

In [ ]:
def get_sentiment(review):

    # TO DO: Complete the prompt below, i.e. fill in the "... ..." between "[INST]" and "[/INST]" 
    # What's Phi's instruction format?
    prompt = f"[INST] ... ... [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # TO DO: Modify below (replace None and complete the code)
        output = None

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract only the model's new part (after the [/INST] tag)
    # What's Phi's instruction format?
    sentiment = response.split("[/INST]")[-1].strip()
    return sentiment

# Example Test
print(get_sentiment("This project is quite challenging but the support is great!"))
# Expected Output: Positive

Now we move beyond simple generation to evaluate how Mistral-7B handles semantic complexity. We will compare two fundamental paradigms of In-Context Learning:
- Zero-shot learning: The model is given a task description and a test input with no prior examples. This relies entirely on the model's pre-trained internal knowledge and instruction-following capabilities.
- Few-shot learning: We provide a "context" of a few labeled examples (k-shot) before the test input. This helps the model align with the desired output format and grasp the specific "logic" of the task.

Standard sentiment analysis often relies simple keyword matching (e.g. "good" = positive). However, human language is rarely that simple. The examples below contains linguistic traps including:
- Sarcasm: "Oh great, another 'groundbreaking' superhero movie!"
- Negation/Double negatives: "I can't believe how much I didn't hate this."
- Contrastive conjuctions: "The plot was non-existent... yet I couldn't look away."
- Backhanded compliments: "This is the best example of a terrible movie."

By comparing zero-shot vs. few-shot performance on these examples, you will observe how providing even a tiny bit of context can help the model navigate complex intent.

#### Exercise: Sentiment analysis of "tricky" sentences

Fill in the code below and compare zero-shot and few-shot performance.

**(You can use Phi model for this task, please modify the code if requried)**

In particular, you would want to search the correct instruction formatr for Phi model, which is different from Mistral.

In [ ]:
# 1. Setup the Evaluation Data
tricky_reviews = [
    "I was worried it would be terrible, but it actually wasn't bad at all.",
    "Oh great, another 'groundbreaking' superhero movie that is exactly like the last ten.",
    "The plot was non-existent and the acting was wooden, yet I couldn't look away.",
    "It’s not that the food was cold, it’s just that it wasn't particularly warm either.",
    "If you enjoy watching paint dry, you'll love this movie!",
    "I've had better, but I've certainly had much, much worse.",
    "Everything about this place is perfect, except for the service, the food, and the price.",
    "I can't believe how much I didn't hate this.",
    "This is the best example of a terrible movie I have ever seen.",
    "The only thing more disappointing than the ending was the fact that I paid for a ticket."
]

# 2. Define the Prompt Templates
# 2. Define the Prompt Templates
def get_zero_shot_prompt(review):
    # TO DO: Construct a zero-shot prompt using Mistral's [INST] format.
    # Ensure you tell the model to output ONLY the label.
    # What's Phi's instruction format?
    return f"[INST] Your Prompt Here [/INST]" 

def get_few_shot_prompt(review):
    # TO DO: Construct a few-shot prompt (at least 3 examples).
    # Use the format: Review: "..." -> Label
    # End the prompt with the current review to be classified.
    # What's Phi's instruction format?
    return f"""[INST] Your Few-Shot Context Here [/INST]"""

# 3. Inference Loop
results = []

print("Starting benchmarking...")
for i, review in enumerate(tricky_reviews):
    print(f"Processing Review {i+1}/10...")

    # Run Zero-Shot
    # TO DO: Tokenize the zero-shot prompt and move it to the GPU
    zs_input = None 
    
    # TO DO: Call model.generate. Keep max_new_tokens low (1-5) since we only want a label.
    zs_output = None
    
    zs_label = tokenizer.decode(zs_output[0], skip_special_tokens=True).split("[/INST]")[-1].strip()

    # Run Few-Shot
    fs_input = tokenizer(get_few_shot_prompt(review), return_tensors="pt").to("cuda")
    
    # TO DO: Generate the output for the few-shot input
    fs_output = None
    
    fs_label = tokenizer.decode(fs_output[0], skip_special_tokens=True).split("[/INST]")[-1].strip()

    results.append({
        "Review": review[:50] + "...", 
        "Zero-Shot Label": zs_label,
        "Few-Shot Label": fs_label
    })

# 4. Display as a Table
df = pd.DataFrame(results)
print("\n--- Sentiment Analysis Results ---")
print(df.to_string(index=False))

# Optional: Save to CSV for your project report
df.to_csv("sentiment_results.csv", index=False)


**Questions (for project report)**

- Did the model respect the instrcution to output only one word in zero-shot? If not, why?
- Identify a specific review where zero-shot failed but few-shot succeeded. What did the model learn from the context?
- How many toekens is your few-shot prompt compared to your zero-shot prompt? When the computing resources are limited, why might we prefer zero-shot for very high-throughput tasks?

## Part 4: Efficient LLM - Compression

In the previous parts, you loaded a 7-billion parameter model in 16-bit precision, requiring about 14.5GB of VRAM. While this fits on a high-end GPU (like an A100), it is far too large for consumer hardware or mobile devices. 

In this section, we explore **quantization**, the process of reducing the precision of the model's weights to shrink its footprint while maintaining as much "intelligence" as possible.

### The mathematics of bits
A standard `bfloat16` ises 16 bits per parameter. Our goal is to compress this to 4 bits per parameter.

Mathematically, this is a lossy compression because we are mapping a continuous range of floating-point values into a small, discrete set of 16 possible levels (2<sup>4</sup> = 16).

But how do we represent a 7B parameters model using only 16 possible values per weight without impacting the model performance (too much)? We use 4-bit NormalFloat (NF4), which uses a non-linear distribution that allocates more "levels" to the most common weight values near zero, perserving the information that matters most to the model's performance.

### 4.1 Implementation: bitsandbytes
We will use the `bitsandbytes` library to perform "Quantization on the fly". Instead of loading 14GB then shrinking it, we will load the model directly into 4-bit memory.

**Task**: Run the code below to load the Mistral model using `BitsAndBytesConfig`.


**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
# 0. Model ID for v0.3 Instruct
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 1. Load Baseline: 16-bit (bfloat16)
print("Loading Baseline Model (bf16)...")
model_16bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=cache_dir
)

In [ ]:
# 2. Define 4-bit Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load Compressed: 4-bit (NF4)
print("Loading Quantized Model (4-bit NF4)...")
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=cache_dir
)

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

**Questions (for project report)**

- In the last line of `bnb_config`, we set `bnb_4bit_compute_dtype=torch.bfloat16`. Why do we need 16 bits when the weights are 4 bits?

### 4.2 The memory paradox: VRAM vs. Disk Size
In theory, 4-bit is 25% of 16 bits. You would expect the VRAM to drop from 14GB to around 3.5 GB. However, you will likely to still see a usage of VRAM of more than 3.5 GB.

In [ ]:
def get_vram_usage(label):
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"[{label}] Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")

# TO DO: Run this check and answer the discussion question
get_vram_usage("Total Memory with Both Models Loaded")

**Questions (for project report)**

- Why is the VRAM usage more than 3.5 GB? You may search for answers from online resources.

In [ ]:
# Helper function to clear VRAM between loads if needed.
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

### Verify the quantization reality with disk size

To truly understand the system-level benefits of quantization, you must compare the static storage. While the 16-bit model is a massive ~14.5GB file, the 4-bit compressed version should represent a near-linear reduction in disk space.

**Exercise**: Export the 16-bit and 4-bit models to your project folder and verify the compression size. We have provided the function of calculating directory size below.

In [ ]:
# Calculate the size of the saved directory in GB
def get_dir_size_gb(directory):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(directory):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024**3)



**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
# TO DO: Save both models to your project directory
path_bf16 = "./models/mistral_bf16"
path_nf4 = "./models/mistral_nf4"

model_16bit.save_pretrained(path_bf16)
model_4bit.save_pretrained(path_nf4)

In [ ]:
# TO DO: Calculate the directory size of each model
print(f"Raw 16-bit Model Disk Size: {get_dir_size_gb(path_bf16):.2f} GB")
print(f"Raw 4-bit Model Disk Size: {get_dir_size_gb(path_nf4):.2f} GB")

### 4.3 Performance benchmarking

Reducing a model from 16-bit to 4-bit is a lossy process. While NF4 is designed to minimize this loss, we should analyse the impact using two methods: mathematical certainity (perplexity) and output quality (human evaluation).

#### 4.3.1 What is perplexity (PPL)?

Mathematically, PPL is the exponentiated average negative log-likelihood of a sequence. If you have a sequence of tokens $X = (x_1, x_2, \dots, x_n)$, the formula is:
$$PPL(X) = \exp \left( -\frac{1}{n} \sum_{i=1}^{n} \log P(x_i | x_{<i}) \right)$$

PPL measures how "surprised" the model is after seeing a sequence of text.

- A low PPL means the model finds the text predictable and highly probable, where it understands the patterns.
- A high PPL means the model is uncertain or confused.

When we quantize a model, we expected the PPL to increase slightly. Our goal is to ensure this "quantization tax" remains low enough that the model's actual utility will not be compromised.

For a deeper dive into how to calculate PPL, please see https://huggingface.co/docs/transformers/perplexity.

#### 4.3.2 Exercise - perplexity

Write a function to calculate the perplexity of a given string. Then run this function on both `model_16bit` and `model_4bit`.


**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    """
    Calculates the perplexity of a given text for a Causal Language Model.
    """
    # 1. Tokenize the input and move to GPU
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        # 2. Get the model output (ensure you pass labels to get the loss)
        # TO DO: Call the model and pass input_ids as labels
        outputs = None 
        
        # 3. Extract the CrossEntropy loss
        # TO DO: Retrieve the loss from the model output
        loss = None 

    # 4. Calculate Perplexity: exp(loss)
    # TO DO: Implement the final PPL calculation
    perplexity = None 
    
    return perplexity

# Test sample from WikiText-2 (or any high-quality Wikipedia snippet)
wiki_sample = "The University of Melbourne is a public research university located in Melbourne, Australia. Founded in 1853, it is Australia's second oldest university."

ppl_16 = calculate_perplexity(model_16bit, tokenizer, wiki_sample)
ppl_4 = calculate_perplexity(model_4bit, tokenizer, wiki_sample)

print(f"--- Perplexity Results ---")
print(f"16-bit PPL: {ppl_16:.4f}")
print(f"4-bit PPL:  {ppl_4:.4f}")

**Questions (for project report)**

Document your findings in your report.

#### 4.3.3 Exercise - text generation quality

Perplexity only tells us about the model's internal probability distribution. It doesn't tell us if the model has started hallucinating or lost its ability to handle sarcasm (e.g. part 3b).

Use the following sarcastic review from part 3b and ask both models to explain why the review is sarcastic. Compare the quality in terms of coherence, formatting, factual arruracy of the two outputs. You may also use any other texts (not limited to sarcastic review) and metrics to evaluate the output.


**(You can use Phi model for this task, please modify the code if requried)**

In particular, you would want to search the correct instruction formatr for Phi model, which is different from Mistral.


In [ ]:
# Recall that the [INST] tag below is from Mistral's instruction format. What's Phi's instruction format?
sarcastic_prompt = "[INST] Explain why this review is sarcastic: 'If you enjoy watching paint dry, you\'ll love this movie!' [/INST]"

def generate_answer(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("[/INST]")[-1].strip()

print("--- 16-bit Explanation ---")
print(generate_answer(model_16bit, tokenizer, sarcastic_prompt))

print("\n--- 4-bit Explanation ---")
print(generate_answer(model_4bit, tokenizer, sarcastic_prompt))

**Questions (for project report)**

Document your findings in your report.

## Part 5: Efficient LLM - Parameter Efficient Fine-Tuning (PEFT)

Training a LLM is generally divided into two stages, pre-training and fine-tuning.

Even though fine-tuning uses less data than pre-training, a full fine-tuning still requires updating 7 billion parameters of the Mistral model. This means all gradients and optimizer states for every single weight must be stored, which could trip the VRAM requirements that increases from 15GB to over 40GB.

Parameter-Efficient Fine-Tuning (PEFT) allows to achieve the performance of full fine-tuning while only updating a tiny fraction (<1%) of the model's weights.

One of the most powerful PEFT methods is Low-Rank Adaptation (LoRA). The mathematical insight behind LoRA is that when a model is adapted to a new task, the change in the weight matrices is "low-rank". Instead of modifying the giant original weight matrix, we freeze it and learn two much smaller matrices that represent the change.

By using LoRA, we significantly reduce the "trainable parameters". This lowers the memory overhead of the optimizer, allowing to train a 7B model on a single GPU that would otherwise crash during standard fine-tuning.

**Questions (for project report)**

Search for online resources that explain and compare pre-training and fine-tuning. Document your findings in your report.

### 5.1 Reading: The LoRA Hypothesis

Before we implement LoRA, we need to understand the theoretical foundation of this technique. Please read the original paper here: https://arxiv.org/abs/2106.09685

Pay close attention to Section 4, where the authors describe the 'intrinsic dimensionality' of pre-trained models. The core hypothesis is that while the model has billions of parameters, the actual 'knowledge update' required for a new task can be represented in a much lower-dimensional space.

**Note**: Matrix manipulation is common in LoRA (and LLM research and implementation). You may want to understand how these matrices form the LLM architecture on a high level, which is examinable during oral exams.


### 5.2 Implementing standard LoRA (16-bit)

We begin by applying LoRA to our 16-bit model from Part 2. This allows you to observe the architectural change with the addition of "adapter" matrices. You may vary parameters like `r` and `lora_alpha`.

**(You can use Phi model for this task, please modify the code if requried)**

In [ ]:
from peft import LoraConfig, get_peft_model

# 1. Define the LoRA Configuration
# 'r' (Rank) is the key hyperparameter from the paper.
# 'target_modules' identifies which weight matrices we are 'adapting'.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap your 16-bit baseline model
# The original model weights are now 'frozen' (requires_grad = False)
model = model_raw #from part 2
peft_model_16bit = get_peft_model(model, lora_config)

# 3. Inspect the trainable parameters
peft_model_16bit.print_trainable_parameters()

### 5.3 Implementing the QLoRA training loop

While LoRA reduces the number of trainable parameters, the base Mistral model still occupies 14GB of VRAM. QLoRA (Quantized LoRA) attaches these adapters to the 4-bit model from Part 4. This allows for full training loops on consumer-grade GPUs.

**(You can use Phi model for this task, please modify the code if requried)**

We will run a mini-funetuning session using a subset of the IMDB dataset.

In [ ]:
dataset = load_dataset("imdb", cache_dir=cache_dir)

In [ ]:
# Rename 'label' to 'labels' to match Mistral's forward() signature
dataset = dataset.rename_column("label", "labels")

# Verify the change
print(dataset['train'].column_names)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training

# 1. Prepare 4-bit model for gradient updates
model_4bit_prepared = prepare_model_for_kbit_training(model_4bit)
qlora_model = get_peft_model(model_4bit_prepared, lora_config)
tokenizer.pad_token = tokenizer.eos_token

# 2. Data Preparation: [INST] Template Formatting
small_train_ds = dataset['train'].shuffle(seed=42).select(range(100))

def preprocess(examples):
    texts = [f"[INST] Review: {t} [/INST] Sentiment: {'Positive' if l==1 else 'Negative'}"
             for t, l in zip(examples['text'], examples['labels'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=256)

tokenized_ds = small_train_ds.map(preprocess, batched=True)

# 3. The Training Loop (Optimized for GPUs)
training_args = TrainingArguments(
    output_dir="./qlora_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    fp16=True,
    logging_steps=5,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Executing QLoRA Training...")
trainer.train()

In [ ]:
# TO DO: Evaulate your model after fine-tuning with QLoRA

**Questions (for project report)**

- Report the exact percentage of parameters you trained. How does this compare to the 7 billion total?

- Record the peak VRAM usage during the trainer.train() call. Can this training run be performed on a standard laptop GPU (eg. 8GB VRAM)?

- Save your adapter using `qlora_model.save_pretrained()`. Compare the size of the saved .safetensors file to the original 14 GB Mistral baseline (or the Phi baseline).

- Evaluate your model with perplexity and text generation quality. How's the performance with QLoRA?

## Part 6: Design your own mini project

Refer to the project handout (pdf) for example mini projects and grading rubric.

In [ ]:
# Your mini project starts here